# HSTU on KuaiRand — Colab driver

This notebook only clones the repo and runs the scripts, so the same code path
runs here and locally. Set `Runtime > Change runtime type > GPU` (A100 or L4).

**Stage 0**: GPU check, dataset download, schema validation, dataset profile.

**Stage 1**: sequence construction — action encoding, filtering, temporal split, id remapping.

**Stage 2**: official HSTU encoder on those sequences (item + 7-class action, relative time). No MovieLens.

In [ ]:
#@title Clone the repo { display-mode: "form" }
REPO_URL = "https://github.com/rayzhao27/hstu-shortvideo-rec.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

import os

if os.path.isdir("/content/hstu-shortvideo-rec"):
    !cd /content/hstu-shortvideo-rec && git pull --ff-only
else:
    !git clone --branch {BRANCH} {REPO_URL} /content/hstu-shortvideo-rec

%cd /content/hstu-shortvideo-rec
!pip install -q -r requirements.txt

In [ ]:
!nvidia-smi
!python -m utils.check_env

## Download the data and profile it

Downloads KuaiRand-Pure (45MB archive, 194MB extracted) into `datasets/raw/`,
verifies its md5, prints the schema and the profile, then writes
`datasets/processed/stats.json`, the parquet caches and `pictures/*.png`.

`/content` is wiped when the runtime restarts. To keep the data, mount Drive:

```python
from google.colab import drive
drive.mount("/content/drive")
```

and add `--raw-dir /content/drive/MyDrive/hstu/raw` below.

In [ ]:
!python -m data.explore --splits standard random

In [ ]:
#@title Stage 0 acceptance numbers and figures
import json
from pathlib import Path

from IPython.display import Image, display

stats = json.loads(Path("datasets/processed/stats.json").read_text())
for split, s in stats.items():
    print(f"{split:9s} users={s['n_users']:>8,}  items={s['n_items']:>7,}  "
          f"interactions={s['n_interactions']:>10,}  mean_seq_len={s['mean_seq_len']:.1f}")

for png in sorted(Path("pictures").glob("stage0_*.png")):
    display(Image(filename=str(png)))

## Stage 1 — build the sequences

Merges the standard and random logs, drops duplicate impressions, encodes each
impression as one action, runs k-core filtering, cuts train/val/test by time, and
remaps the ids. Writes `{train,val,test}_seqs.pkl`, the encoders,
`preprocess_stats.json` and `pictures/stage1_*.png`.

Useful variations:

```bash
!python -m data.preprocess --split-strategy loo          # leave-last-one-out
!python -m data.preprocess --target-policy recommended   # score only is_rand=0
!python -m data.preprocess --no-random                   # biased log only
```

In [ ]:
!python -m data.preprocess
!python -m data.verify
!python -m data.protocol

In [ ]:
#@title Stage 1 figures and a sample sequence
import json
from pathlib import Path

from IPython.display import Image, display

from data.actions import ACTION_NAMES
from data.encoders import IdEncoder
from data.sequences import load_sequences

for png in sorted(Path("pictures").glob("stage1_*.png")):
    display(Image(filename=str(png)))

item_encoder = IdEncoder.load(Path("datasets/processed/item_encoder.pkl"))
test = load_sequences(Path("datasets/processed/test_seqs.pkl"))

# A user with a typical amount of history, shown around their first test target.
record = sorted(test, key=lambda r: len(r["items"]))[len(test) // 2]
first_target = int(record["is_target"].argmax())
print(f"user {record['user']}: {len(record['items'])} interactions, "
      f"{record['n_targets']} test targets\n")
for i in range(max(0, first_target - 5), min(len(record["items"]), first_target + 5)):
    print(f"  {'>' if record['is_target'][i] else ' '} "
          f"item {record['items'][i]:>5} (raw {item_encoder.idx_to_raw[record['items'][i]]:>5})  "
          f"{ACTION_NAMES[record['actions'][i]]:<10} "
          f"{'random' if record['is_rand'][i] else 'recommended'}")

## Stage 2 — HSTU smoke test

Clones Meta's `generative-recommenders` into `third_party/` (not our data),
installs the Python deps the official README names, then runs one batch of
**our** Stage 1 sequences through the official HSTU encoder.

`fbgemm_gpu` / `torchrec` only resolve on Ubuntu + CUDA + Python 3.10. If the
wheel does not match this runtime, `models.smoke` falls back to a dense
implementation of the three fbgemm ops and still checks forward + backward.

In [ ]:
!python -m utils.meta_repo
!pip install -q gin-config tensorboard
# Official extra deps; ignore a wheel mismatch — smoke has a fallback.
!pip install -q fbgemm-gpu torchrec || true
!python -m models.smoke --device cuda